# Prepare Calendar Table

Generates a standard date dimension table `adb_hospital.silver.calender` spanning **1940-01-01 to 2030-12-31** with all common calendar attributes.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DateType

# Generate date sequence from 1940-01-01 to 2030-12-31
df_dates = (
    spark.sql("SELECT sequence(DATE'1940-01-01', DATE'2030-12-31', INTERVAL 1 DAY) AS date_array")
    .select(F.explode("date_array").alias("date"))
)

# Build all standard calendar columns
df_calendar = df_dates.select(
    # --- Key & core date ---
    F.date_format("date", "yyyyMMdd").cast("int").alias("date_key"),
    F.col("date"),

    # --- Year ---
    F.year("date").alias("year"),

    # --- Quarter ---
    F.quarter("date").alias("quarter"),
    F.concat(F.lit("Q"), F.quarter("date")).alias("quarter_name"),

    # --- Month ---
    F.month("date").alias("month"),
    F.date_format("date", "MMMM").alias("month_name"),
    F.date_format("date", "MMM").alias("month_short_name"),

    # --- Week ---
    F.weekofyear("date").alias("week_of_year"),

    # --- Day ---
    F.dayofmonth("date").alias("day_of_month"),
    F.dayofweek("date").alias("day_of_week"),        # 1=Sun … 7=Sat
    F.dayofyear("date").alias("day_of_year"),
    F.date_format("date", "EEEE").alias("day_name"),
    F.date_format("date", "EEE").alias("day_short_name"),

    # --- Weekend / Weekday ---
    F.when(F.dayofweek("date").isin(1, 7), True).otherwise(False).alias("is_weekend"),
    F.when(F.dayofweek("date").isin(1, 7), "Weekend").otherwise("Weekday").alias("day_type"),

    # --- Start / End of period ---
    F.trunc("date", "week").alias("start_of_week"),
    F.trunc("date", "month").alias("start_of_month"),
    F.trunc("date", "quarter").alias("start_of_quarter"),
    F.trunc("date", "year").alias("start_of_year"),
    F.last_day("date").alias("end_of_month"),

    # --- Year-month label ---
    F.date_format("date", "yyyy-MM").alias("year_month"),
    F.date_format("date", "yyyy-'Q'").alias("year_quarter_prefix"),
    F.concat(F.year("date"), F.lit("-Q"), F.quarter("date")).alias("year_quarter"),

    # --- Fiscal calendar (July = FY start) ---
    F.when(F.month("date") >= 7, F.year("date") + 1).otherwise(F.year("date")).alias("fiscal_year"),
    F.ceil(
        F.when(F.month("date") >= 7, F.month("date") - 6)
         .otherwise(F.month("date") + 6) / 3
    ).cast("int").alias("fiscal_quarter"),

    # --- Leap year ---
    F.when(
        (F.year("date") % 4 == 0) &
        ((F.year("date") % 100 != 0) | (F.year("date") % 400 == 0)),
        True
    ).otherwise(False).alias("is_leap_year"),

    # --- Days in month ---
    F.dayofmonth(F.last_day("date")).alias("days_in_month"),

    # --- Month / Quarter progress ---
    F.when(
        F.dayofmonth("date") <= 10, "First"
    ).when(
        F.dayofmonth("date") <= 20, "Second"
    ).otherwise("Third").alias("month_third"),

    F.when(
        F.dayofmonth("date") <= 15, "First Half"
    ).otherwise("Second Half").alias("month_half"),

    F.when(
        F.month("date") <= 6, "H1"
    ).otherwise("H2").alias("year_half"),
)

# Write to silver layer
df_calendar.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("adb_hospital.silver.calender")

print(f"Calendar table written — {df_calendar.count():,} rows")
display(spark.table("adb_hospital.silver.calender").limit(10))